# 🔮 Self-Adaptive Homomorphic Framework with ML-Driven Noise Prediction for Confidential AI Computation (TensorFlow / Keras)
> **Full Pipeline:** Dataset Generation → Preprocessing → Model Training → Evaluation → Visualization → Export to TensorFlow.js

---

## 📋 Table of Contents
1. [Install Dependencies](#1)
2. [Import Libraries](#2)
3. [Generate Dataset](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Preprocess Data](#5)
6. [Build LSTM Model](#6)
7. [Train Model](#7)
8. [Evaluate & Predict](#8)
9. [Visualize Results](#9)
10. [Export to TensorFlow.js](#10)
11. [Use Model in Website](#11)

---
## 1. 📦 Install Dependencies <a name="1"></a>

In [2]:
# Install required packages
!pip install tensorflowjs --quiet
!pip install matplotlib seaborn scikit-learn pandas numpy --quiet

print("✅ All packages installed successfully!")

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [18 lines of output]
      Traceback (most recent call last):
        File "C:\Users\hariv\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "C:\Users\hariv\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\hariv\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_proces

✅ All packages installed successfully!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\hariv\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


---
## 2. 📚 Import Libraries <a name="2"></a>

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os
import json

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, BatchNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam

import tensorflowjs as tfjs

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"✅ TensorFlow version : {tf.__version__}")
print(f"✅ TensorFlow.js version: {tfjs.__version__}")
print(f"✅ NumPy version      : {np.__version__}")

ModuleNotFoundError: No module named 'tensorflowjs'

---
## 3. 🗄️ Generate Dataset <a name="3"></a>

We simulate a **realistic stock-price-like time series** that combines:
- 📈 Long-term upward trend
- 🔄 Seasonal / cyclical patterns (weekly + yearly)
- 🎲 Random Gaussian noise
- ⚡ Occasional shock events

This mimics real-world scenarios (energy demand, sales, temperature, stock prices).

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
N_DAYS        = 1460          # 4 years of daily data
WINDOW_SIZE   = 60            # Look-back window (days)
FORECAST_HORIZON = 1          # Steps ahead to predict
TEST_SPLIT    = 0.15          # 15% test set
VAL_SPLIT     = 0.15          # 15% validation set

# ── Generate synthetic time series ────────────────────────────────────────────
date_range = pd.date_range(start='2020-01-01', periods=N_DAYS, freq='D')
t = np.arange(N_DAYS)

trend       = 0.05  * t                                       # Linear trend
yearly_cycle= 15    * np.sin(2 * np.pi * t / 365.25)          # Yearly seasonality
weekly_cycle= 5     * np.sin(2 * np.pi * t / 7)               # Weekly seasonality
noise       = np.random.normal(0, 3, N_DAYS)                   # Gaussian noise

# Random shock events (simulate market crashes / spikes)
shocks = np.zeros(N_DAYS)
shock_idx = np.random.choice(N_DAYS, size=15, replace=False)
shocks[shock_idx] = np.random.choice([-20, -15, 20, 25], size=15)

base_value  = 100
series      = base_value + trend + yearly_cycle + weekly_cycle + noise + shocks
series      = np.clip(series, 50, None)   # Prices can't go below 50

# ── Create DataFrame ──────────────────────────────────────────────────────────
df = pd.DataFrame({'Date': date_range, 'Value': series})
df.set_index('Date', inplace=True)

# Add derived features
df['Rolling_Mean_7']  = df['Value'].rolling(7).mean()
df['Rolling_Std_7']   = df['Value'].rolling(7).std()
df['Rolling_Mean_30'] = df['Value'].rolling(30).mean()
df['Lag_1']           = df['Value'].shift(1)
df['Lag_7']           = df['Value'].shift(7)
df['Lag_30']          = df['Value'].shift(30)
df['Day_of_Week']     = df.index.dayofweek
df['Month']           = df.index.month
df['Quarter']         = df.index.quarter
df.dropna(inplace=True)

print(f"📊 Dataset shape      : {df.shape}")
print(f"📅 Date range         : {df.index.min().date()} → {df.index.max().date()}")
print(f"📈 Value range        : {df['Value'].min():.2f} → {df['Value'].max():.2f}")
print(f"\n📋 First 5 rows:")
df.head()

In [ ]:
# Save dataset to CSV
df.to_csv('time_series_dataset.csv')
print("✅ Dataset saved to time_series_dataset.csv")
print(f"\n📝 Dataset Statistics:")
df[['Value','Rolling_Mean_7','Rolling_Mean_30']].describe().round(2)

---
## 4. 🔍 Exploratory Data Analysis (EDA) <a name="4"></a>

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Time Series – Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

# 1. Full time series
ax = axes[0, 0]
ax.plot(df.index, df['Value'], color='steelblue', linewidth=0.8, alpha=0.8, label='Daily Value')
ax.plot(df.index, df['Rolling_Mean_7'], color='orange', linewidth=1.5, label='7-day MA')
ax.plot(df.index, df['Rolling_Mean_30'], color='red', linewidth=1.5, label='30-day MA')
ax.set_title('Full Time Series with Moving Averages')
ax.set_xlabel('Date'); ax.set_ylabel('Value')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# 2. Distribution
ax = axes[0, 1]
ax.hist(df['Value'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['Value'].mean(), color='red', linestyle='--', label=f"Mean={df['Value'].mean():.1f}")
ax.axvline(df['Value'].median(), color='orange', linestyle='--', label=f"Median={df['Value'].median():.1f}")
ax.set_title('Value Distribution')
ax.set_xlabel('Value'); ax.set_ylabel('Frequency')
ax.legend()

# 3. Seasonal – Day of week
ax = axes[1, 0]
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_means = df.groupby('Day_of_Week')['Value'].mean()
ax.bar(dow_labels, dow_means, color=sns.color_palette('viridis', 7))
ax.set_title('Average Value by Day of Week')
ax.set_xlabel('Day'); ax.set_ylabel('Avg Value')

# 4. Seasonal – Month
ax = axes[1, 1]
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_means = df.groupby('Month')['Value'].mean()
ax.bar(month_labels, month_means, color=sns.color_palette('magma', 12))
ax.set_title('Average Value by Month')
ax.set_xlabel('Month'); ax.set_ylabel('Avg Value')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# 5. Rolling volatility
ax = axes[2, 0]
ax.plot(df.index, df['Rolling_Std_7'], color='purple', linewidth=0.8)
ax.fill_between(df.index, 0, df['Rolling_Std_7'], alpha=0.3, color='purple')
ax.set_title('7-day Rolling Volatility (Std Dev)')
ax.set_xlabel('Date'); ax.set_ylabel('Std Dev')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# 6. Autocorrelation (manual)
ax = axes[2, 1]
lags = range(1, 61)
autocorr = [df['Value'].autocorr(lag=l) for l in lags]
ax.bar(lags, autocorr, color='teal', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.2, color='red', linestyle='--', linewidth=0.8, label='±0.2 threshold')
ax.axhline(-0.2, color='red', linestyle='--', linewidth=0.8)
ax.set_title('Autocorrelation (Lags 1–60)')
ax.set_xlabel('Lag (days)'); ax.set_ylabel('Autocorrelation')
ax.legend()

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight')
plt.show()
print("✅ EDA plots saved to eda_plots.png")

---
## 5. ⚙️ Preprocess Data <a name="5"></a>

Steps:
1. **Scale** features to [0, 1] using MinMaxScaler
2. **Create sliding windows** — input sequence of `WINDOW_SIZE` timesteps → predict next value
3. **Split** into train / validation / test sets (chronologically)

In [ ]:
# ── Feature selection ─────────────────────────────────────────────────────────
FEATURES = ['Value', 'Rolling_Mean_7', 'Rolling_Mean_30',
            'Lag_1', 'Lag_7', 'Day_of_Week', 'Month']
TARGET   = 'Value'

data = df[FEATURES].values
target_col_idx = FEATURES.index(TARGET)

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Separate scaler for target only (needed for inverse transform)
target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaled = target_scaler.fit_transform(df[[TARGET]].values)

# ── Create sliding windows ────────────────────────────────────────────────────
def create_sequences(data, target, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i])          # shape: (window, n_features)
        y.append(target[i, 0])                 # next step target value
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, target_scaled, WINDOW_SIZE)

# ── Chronological train/val/test split ────────────────────────────────────────
total   = len(X)
n_test  = int(total * TEST_SPLIT)
n_val   = int(total * VAL_SPLIT)
n_train = total - n_test - n_val

X_train, y_train = X[:n_train],             y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],        y[n_train+n_val:]

print(f"📐 Input shape          : (samples, timesteps, features) = {X.shape}")
print(f"\n🔀 Split summary:")
print(f"   Train  : {X_train.shape[0]:,} samples")
print(f"   Val    : {X_val.shape[0]:,} samples")
print(f"   Test   : {X_test.shape[0]:,} samples")
print(f"\n📏 Window size          : {WINDOW_SIZE} days")
print(f"📊 Number of features   : {len(FEATURES)}")

---
## 6. 🧠 Build LSTM Model <a name="6"></a>

Architecture: **Stacked Bidirectional LSTM** with:
- Bidirectional LSTM layers (captures both past and future context in training)
- Batch Normalization (stabilizes training)
- Dropout (prevents overfitting)
- Dense output layer

In [ ]:
n_features = X_train.shape[2]

def build_model(window_size, n_features, units=64, dropout=0.2):
    model = Sequential([
        # Layer 1 – Bidirectional LSTM
        Bidirectional(
            LSTM(units, return_sequences=True),
            input_shape=(window_size, n_features)
        ),
        BatchNormalization(),
        Dropout(dropout),

        # Layer 2 – LSTM
        LSTM(units // 2, return_sequences=True),
        BatchNormalization(),
        Dropout(dropout),

        # Layer 3 – LSTM
        LSTM(units // 4, return_sequences=False),
        BatchNormalization(),
        Dropout(dropout),

        # Dense head
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)  # Single-step forecast
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='huber',                   # Robust to outliers
        metrics=['mae']
    )
    return model

model = build_model(WINDOW_SIZE, n_features)
model.summary()

total_params = model.count_params()
print(f"\n🔢 Total parameters: {total_params:,}")

---
## 7. 🏋️ Train Model <a name="7"></a>

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        'best_model.keras', monitor='val_loss',
        save_best_only=True, verbose=0
    ),
]

# ── Train ──────────────────────────────────────────────────────────────────────
print("🚀 Training started...\n")
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print(f"\n✅ Training complete!")
print(f"   Best val_loss : {min(history.history['val_loss']):.6f}")
print(f"   Best val_mae  : {min(history.history['val_mae']):.6f}")
print(f"   Epochs run    : {len(history.history['loss'])}")

In [ ]:
# ── Plot training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(history.history['loss'],     label='Train Loss', color='steelblue')
ax.plot(history.history['val_loss'], label='Val Loss',   color='coral')
ax.set_title('Huber Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend()

# MAE
ax = axes[1]
ax.plot(history.history['mae'],     label='Train MAE', color='steelblue')
ax.plot(history.history['val_mae'], label='Val MAE',   color='coral')
ax.set_title('Mean Absolute Error')
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (scaled)')
ax.legend()

plt.tight_layout()
plt.savefig('training_history.png', bbox_inches='tight')
plt.show()
print("✅ Training history plot saved.")

---
## 8. 📊 Evaluate & Predict <a name="8"></a>

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_train_scaled = model.predict(X_train, verbose=0)
y_pred_val_scaled   = model.predict(X_val,   verbose=0)
y_pred_test_scaled  = model.predict(X_test,  verbose=0)

# Inverse transform to original scale
y_pred_train = target_scaler.inverse_transform(y_pred_train_scaled)
y_pred_val   = target_scaler.inverse_transform(y_pred_val_scaled)
y_pred_test  = target_scaler.inverse_transform(y_pred_test_scaled)

y_true_train = target_scaler.inverse_transform(y_train.reshape(-1, 1))
y_true_val   = target_scaler.inverse_transform(y_val.reshape(-1, 1))
y_true_test  = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"📌 {label:10s} | MAE: {mae:6.3f}  RMSE: {rmse:6.3f}  R²: {r2:.4f}  MAPE: {mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

print("═" * 65)
print(f"{'METRIC SUMMARY':^65}")
print("═" * 65)
train_metrics = compute_metrics(y_true_train, y_pred_train, 'Train')
val_metrics   = compute_metrics(y_true_val,   y_pred_val,   'Validation')
test_metrics  = compute_metrics(y_true_test,  y_pred_test,  'Test')
print("═" * 65)

# Save metrics to JSON
metrics_dict = {'train': train_metrics, 'val': val_metrics, 'test': test_metrics}
with open('metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print("\n✅ Metrics saved to metrics.json")

In [ ]:
# ── Multi-step future forecast ─────────────────────────────────────────────────
FUTURE_STEPS = 30  # Forecast 30 days into the future

# Use the last window from the test set as seed
last_window = data_scaled[-WINDOW_SIZE:].copy()   # shape: (60, n_features)
future_preds = []

for _ in range(FUTURE_STEPS):
    inp = last_window.reshape(1, WINDOW_SIZE, n_features)
    pred = model.predict(inp, verbose=0)[0, 0]
    future_preds.append(pred)

    # Roll window: drop oldest, append new prediction in target column
    new_row = last_window[-1].copy()
    new_row[target_col_idx] = pred
    last_window = np.vstack([last_window[1:], new_row])

# Inverse transform
future_preds_arr = np.array(future_preds).reshape(-1, 1)
future_values    = target_scaler.inverse_transform(future_preds_arr).flatten()

last_date         = df.index[-1]
future_dates      = pd.date_range(start=last_date + pd.Timedelta(days=1),
                                   periods=FUTURE_STEPS, freq='D')

print(f"✅ {FUTURE_STEPS}-day future forecast generated")
print(f"   Forecast range: {future_dates[0].date()} → {future_dates[-1].date()}")
print(f"   Predicted range: {future_values.min():.2f} → {future_values.max():.2f}")

---
## 9. 📈 Visualize Results <a name="9"></a>

In [ ]:
# ── Reconstruct dates for each split ──────────────────────────────────────────
dates_all   = df.index[WINDOW_SIZE:]
dates_train = dates_all[:n_train]
dates_val   = dates_all[n_train:n_train + n_val]
dates_test  = dates_all[n_train + n_val:]

fig, axes = plt.subplots(3, 1, figsize=(16, 16))
fig.suptitle('LSTM Time Series Forecast Results', fontsize=16, fontweight='bold')

# ── 1. Full overview ──────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(dates_train, y_true_train, color='steelblue',  lw=0.7, alpha=0.6, label='Train (Actual)')
ax.plot(dates_val,   y_true_val,   color='orange',      lw=0.7, alpha=0.6, label='Val (Actual)')
ax.plot(dates_test,  y_true_test,  color='green',       lw=0.7, alpha=0.6, label='Test (Actual)')
ax.plot(dates_train, y_pred_train, color='steelblue',  lw=1.2, linestyle='--', label='Train (Pred)')
ax.plot(dates_val,   y_pred_val,   color='orange',      lw=1.2, linestyle='--', label='Val (Pred)')
ax.plot(dates_test,  y_pred_test,  color='green',       lw=1.2, linestyle='--', label='Test (Pred)')
# Shade regions
ax.axvspan(dates_val[0], dates_val[-1],   alpha=0.08, color='orange', label='Val region')
ax.axvspan(dates_test[0], dates_test[-1], alpha=0.08, color='green',  label='Test region')
ax.set_title('Full Series: Actual vs Predicted')
ax.set_xlabel('Date'); ax.set_ylabel('Value')
ax.legend(ncol=3, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# ── 2. Test set close-up + future forecast ────────────────────────────────────
ax = axes[1]
ax.plot(dates_test, y_true_test, color='green',  lw=1.5, label='Test Actual')
ax.plot(dates_test, y_pred_test, color='red',    lw=1.5, linestyle='--', label='Test Predicted')
ax.fill_between(dates_test,
                y_true_test.flatten(), y_pred_test.flatten(),
                alpha=0.15, color='red', label='Error')
# Future forecast
ax.plot(future_dates, future_values, color='purple', lw=2, linestyle=':', marker='o',
        markersize=4, label=f'{FUTURE_STEPS}-day Forecast')
ax.axvline(x=future_dates[0], color='purple', linestyle='--', alpha=0.5, label='Forecast start')
ax.fill_between(future_dates,
                future_values * 0.97, future_values * 1.03,
                alpha=0.2, color='purple', label='±3% CI')
ax.set_title(f'Test Set + {FUTURE_STEPS}-Day Future Forecast')
ax.set_xlabel('Date'); ax.set_ylabel('Value')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# ── 3. Residuals plot ─────────────────────────────────────────────────────────
ax = axes[2]
residuals = y_true_test.flatten() - y_pred_test.flatten()
ax.bar(range(len(residuals)), residuals,
       color=np.where(residuals >= 0, 'steelblue', 'coral'), alpha=0.7, width=1)
ax.axhline(0, color='black', lw=1)
ax.axhline(residuals.mean() + 2*residuals.std(), color='red',
           linestyle='--', lw=0.8, label='+2σ')
ax.axhline(residuals.mean() - 2*residuals.std(), color='red',
           linestyle='--', lw=0.8, label='-2σ')
ax.set_title('Test Set Residuals (Actual − Predicted)')
ax.set_xlabel('Test Sample Index'); ax.set_ylabel('Residual')
ax.legend()

plt.tight_layout()
plt.savefig('forecast_results.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Forecast results plot saved to forecast_results.png")

In [ ]:
# ── Metrics dashboard ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Metrics', fontsize=14, fontweight='bold')

sets    = ['Train', 'Validation', 'Test']
colors  = ['steelblue', 'orange', 'green']
metrics = [train_metrics, val_metrics, test_metrics]

for idx, (metric_name, ylabel) in enumerate([('mae','MAE'), ('rmse','RMSE'), ('r2','R² Score')]):
    ax = axes[idx]
    vals = [m[metric_name] for m in metrics]
    bars = ax.bar(sets, vals, color=colors, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
    ax.set_title(ylabel)
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.savefig('metrics_dashboard.png', bbox_inches='tight')
plt.show()
print("✅ Metrics dashboard saved to metrics_dashboard.png")

---
## 10. 🌐 Export to TensorFlow.js <a name="10"></a>

We'll export the trained Keras model to **TensorFlow.js format** so it can run directly in the browser — no server required!

The export produces:
- `model.json` — architecture + metadata
- `group1-shard1of1.bin` — model weights

In [ ]:
# ── Export Keras model to TensorFlow.js ───────────────────────────────────────
TFJS_OUTPUT_DIR = './tfjs_model'
os.makedirs(TFJS_OUTPUT_DIR, exist_ok=True)

tfjs.converters.save_keras_model(model, TFJS_OUTPUT_DIR)

print(f"✅ Model exported to TensorFlow.js format!")
print(f"📁 Output directory: {TFJS_OUTPUT_DIR}")
print("\n📋 Exported files:")
for f in sorted(os.listdir(TFJS_OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(TFJS_OUTPUT_DIR, f))
    print(f"   {f:<40} {size/1024:.1f} KB")

In [ ]:
# ── Save scaler parameters for use in the browser ─────────────────────────────
scaler_params = {
    'features': FEATURES,
    'target': TARGET,
    'window_size': WINDOW_SIZE,
    'feature_min': scaler.data_min_.tolist(),
    'feature_max': scaler.data_max_.tolist(),
    'feature_scale': scaler.scale_.tolist(),
    'target_min': float(target_scaler.data_min_[0]),
    'target_max': float(target_scaler.data_max_[0]),
    'target_scale': float(target_scaler.scale_[0]),
    'metrics': {
        'test_mae':  round(test_metrics['mae'], 4),
        'test_rmse': round(test_metrics['rmse'], 4),
        'test_r2':   round(test_metrics['r2'], 4),
        'test_mape': round(test_metrics['mape'], 2)
    }
}

with open(os.path.join(TFJS_OUTPUT_DIR, 'scaler_params.json'), 'w') as f:
    json.dump(scaler_params, f, indent=2)

print("✅ Scaler parameters saved to tfjs_model/scaler_params.json")
print(f"\n📋 Scaler params preview:")
print(json.dumps({k: v for k, v in scaler_params.items() if k not in ['feature_min','feature_max','feature_scale']}, indent=2))

In [ ]:
# ── Zip and download the model ─────────────────────────────────────────────────
import shutil

shutil.make_archive('tfjs_model_export', 'zip', TFJS_OUTPUT_DIR)

print("✅ Model zipped: tfjs_model_export.zip")
print(f"📦 Zip size: {os.path.getsize('tfjs_model_export.zip') / 1024:.1f} KB")

# Download in Colab
try:
    from google.colab import files
    files.download('tfjs_model_export.zip')
    print("⬇️  Download started!")
except ImportError:
    print("ℹ️  Not in Colab — file saved locally as tfjs_model_export.zip")

---
## 11. 🌍 Use Model in Your Website <a name="11"></a>

After downloading `tfjs_model_export.zip`, extract it and host the files. Below is a **complete ready-to-use HTML page** that:
- Loads TensorFlow.js from CDN
- Loads your exported model
- Accepts 60 days of input values
- Predicts the next day's value in-browser

> 💡 **No server needed** — runs entirely in the browser!

In [ ]:
html_code = '''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>Time Series Forecast – TF.js</title>
  <!-- TensorFlow.js -->
  <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.15.0/dist/tf.min.js"></script>
  <!-- Chart.js for visualization -->
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Segoe UI', sans-serif; background: #0f172a; color: #e2e8f0; min-height: 100vh; padding: 2rem; }
    h1   { font-size: 1.8rem; color: #38bdf8; margin-bottom: 0.5rem; }
    p.sub { color: #94a3b8; margin-bottom: 2rem; }
    .card { background: #1e293b; border-radius: 12px; padding: 1.5rem; margin-bottom: 1.5rem; }
    label { display: block; margin-bottom: 0.5rem; font-weight: 600; color: #cbd5e1; }
    textarea { width: 100%; height: 120px; background: #0f172a; color: #e2e8f0;
               border: 1px solid #334155; border-radius: 8px; padding: 0.75rem; font-size: 0.85rem; }
    button { background: #0ea5e9; color: white; border: none; border-radius: 8px;
             padding: 0.75rem 1.5rem; font-size: 1rem; cursor: pointer; margin-top: 1rem; }
    button:hover { background: #38bdf8; }
    button:disabled { background: #475569; cursor: not-allowed; }
    .result { font-size: 2rem; font-weight: 700; color: #4ade80; margin-top: 1rem; }
    .status { color: #94a3b8; font-size: 0.9rem; margin-top: 0.5rem; }
    canvas  { max-width: 100%; }
  </style>
</head>
<body>
  <h1>🔮 Time Series Forecaster</h1>
  <p class="sub">Powered by LSTM + TensorFlow.js — runs 100% in your browser</p>

  <div class="card">
    <label>📁 Load TF.js Model (select the model.json file)</label>
    <input type="file" id="modelFile" multiple accept=".json,.bin" />
    <p class="status" id="modelStatus">⏳ No model loaded yet</p>
  </div>

  <div class="card">
    <label>📊 Input: Last 60 Daily Values (comma-separated)</label>
    <textarea id="inputValues" placeholder="e.g. 102.3, 103.1, 101.8, ..."></textarea>
    <button id="predictBtn" onclick="predict()" disabled>Predict Next Day ➜</button>
  </div>

  <div class="card" id="resultCard" style="display:none">
    <label>📈 Prediction Result</label>
    <div class="result" id="resultValue"></div>
    <p class="status" id="resultDetail"></p>
    <canvas id="forecastChart" style="margin-top:1.5rem"></canvas>
  </div>

  <script>
    let model = null;
    let scalerParams = null;
    const WINDOW_SIZE = 60;

    // ── Load scaler params alongside model ─────────────────────────────────────
    document.getElementById('modelFile').addEventListener('change', async (e) => {
      const files = Array.from(e.target.files);
      const jsonFile = files.find(f => f.name === 'model.json');
      if (!jsonFile) { alert('Please select model.json'); return; }

      document.getElementById('modelStatus').textContent = '⏳ Loading model...';
      try {
        model = await tf.loadLayersModel(tf.io.browserFiles([jsonFile, ...files.filter(f => f.name.endsWith('.bin'))]));
        document.getElementById('modelStatus').textContent = '✅ Model loaded! Ready to predict.';
        document.getElementById('predictBtn').disabled = false;

        // Try to load scaler_params.json
        const scalerFile = files.find(f => f.name === 'scaler_params.json');
        if (scalerFile) {
          scalerParams = JSON.parse(await scalerFile.text());
          document.getElementById('modelStatus').textContent += ' Scaler loaded.';
        }
      } catch (err) {
        document.getElementById('modelStatus').textContent = '❌ Error: ' + err.message;
      }
    });

    // ── Normalise a single value using MinMaxScaler params ────────────────────
    function scaleValue(val, min, max) {
      return (val - min) / (max - min);
    }
    function inverseScaleValue(val, min, max) {
      return val * (max - min) + min;
    }

    // ── Run prediction ────────────────────────────────────────────────────────
    async function predict() {
      const raw = document.getElementById('inputValues').value
        .split(',').map(v => parseFloat(v.trim())).filter(v => !isNaN(v));

      if (raw.length < WINDOW_SIZE) {
        alert(`Please provide at least ${WINDOW_SIZE} values. You provided ${raw.length}.`); return;
      }

      const window = raw.slice(-WINDOW_SIZE);
      const tMin = scalerParams ? scalerParams.target_min : Math.min(...window);
      const tMax = scalerParams ? scalerParams.target_max : Math.max(...window);

      // Build input tensor [1, 60, n_features] — use Value as first feature
      const nFeatures = scalerParams ? scalerParams.features.length : 1;
      const inputData = [];
      for (let i = 0; i < WINDOW_SIZE; i++) {
        const row = [];
        const fMin = scalerParams ? scalerParams.feature_min : [tMin];
        const fMax = scalerParams ? scalerParams.feature_max : [tMax];
        for (let f = 0; f < nFeatures; f++) {
          const v = (f === 0) ? window[i] : window[Math.max(0, i - f)];
          row.push(scaleValue(v, fMin[f], fMax[f]));
        }
        inputData.push(row);
      }

      const tensor = tf.tensor3d([inputData]);
      const predTensor = model.predict(tensor);
      const predScaled = (await predTensor.data())[0];
      const predValue  = inverseScaleValue(predScaled, tMin, tMax);

      tensor.dispose(); predTensor.dispose();

      const lastVal = window[window.length - 1];
      const change  = predValue - lastVal;
      const changePct = (change / lastVal * 100).toFixed(2);
      const arrow   = change >= 0 ? '▲' : '▼';

      document.getElementById('resultCard').style.display = 'block';
      document.getElementById('resultValue').textContent = `${predValue.toFixed(2)}`;
      document.getElementById('resultDetail').textContent =
        `${arrow} ${Math.abs(change).toFixed(2)} (${changePct}%) vs last value of ${lastVal.toFixed(2)}`;

      drawChart(window, predValue);
    }

    // ── Draw chart ────────────────────────────────────────────────────────────
    let chartInstance = null;
    function drawChart(history, prediction) {
      if (chartInstance) chartInstance.destroy();
      const labels = [...history.map((_, i) => `T-${WINDOW_SIZE - 1 - i}`), 'T+1'];
      const data   = [...history, null];
      const predData = [...Array(history.length).fill(null), prediction];

      chartInstance = new Chart(document.getElementById('forecastChart'), {
        type: 'line',
        data: {
          labels,
          datasets: [
            { label: 'Historical', data, borderColor: '#38bdf8', backgroundColor: 'transparent',
              pointRadius: 2, tension: 0.3 },
            { label: 'Forecast', data: predData, borderColor: '#4ade80', backgroundColor: 'rgba(74,222,128,0.3)',
              pointRadius: 8, pointStyle: 'star', tension: 0 }
          ]
        },
        options: {
          responsive: true,
          plugins: { legend: { labels: { color: '#e2e8f0' } } },
          scales: {
            x: { ticks: { color: '#94a3b8', maxTicksLimit: 12 }, grid: { color: '#1e293b' } },
            y: { ticks: { color: '#94a3b8' }, grid: { color: '#334155' } }
          }
        }
      });
    }
  </script>
</body>
</html>
'''

with open('forecast_website.html', 'w') as f:
    f.write(html_code)

print("✅ Website HTML saved to forecast_website.html")

# Download it
try:
    from google.colab import files
    files.download('forecast_website.html')
    print("⬇️  HTML download started!")
except ImportError:
    print("ℹ️  File saved locally as forecast_website.html")

---
## ✅ Summary

| Step | Description | Output File |
|------|-------------|-------------|
| 1 | Install dependencies | — |
| 2 | Import libraries | — |
| 3 | Generate synthetic dataset | `time_series_dataset.csv` |
| 4 | Exploratory Data Analysis | `eda_plots.png` |
| 5 | Preprocess + create sequences | — |
| 6 | Build Bidirectional LSTM model | — |
| 7 | Train with callbacks | `best_model.keras`, `training_history.png` |
| 8 | Evaluate + 30-day forecast | `metrics.json` |
| 9 | Visualize results | `forecast_results.png`, `metrics_dashboard.png` |
| 10 | Export to TensorFlow.js | `tfjs_model_export.zip` |
| 11 | Ready-to-use website | `forecast_website.html` |

### 🚀 How to deploy on your website
1. Extract `tfjs_model_export.zip` → you get `model.json` + `.bin` + `scaler_params.json`
2. Place all files in the **same folder** as `forecast_website.html`
3. Open `forecast_website.html` in any modern browser — **no server needed!**
4. Click `Load Model`, select `model.json` (and `.bin` files), then enter 60 values and click **Predict**

> 💡 To deploy publicly, host the files on **GitHub Pages**, **Netlify**, or **Vercel** for free!